# Lezione 4 — Probabilità, variabili casuali e valori attesi condizionati in Python

**Versione studente**

Caso applicativo: **rendimenti e perdite di un portafoglio finanziario a un periodo con stati informativi di mercato**.

La lezione usa una simulazione Monte Carlo per tradurre in procedure computabili alcuni oggetti introdotti nei Capitoli 1--3:

- variabile casuale discreta di stato $Z$;
- rendimento aleatorio $R$;
- perdita $L=-V_0R$;
- eventi di superamento soglia;
- probabilità empiriche;
- valori attesi non condizionati;
- valori attesi condizionati rispetto a eventi e partizioni informative;
- costruzione computazionale della variabile $\mathbb{E}[L\mid\mathcal{G}]$.

Il notebook è organizzato per **domande probabilistiche**, non per librerie Python.

Le formule nelle celle di testo usano sintassi Markdown/Jupyter: `$...$` per formule inline e `$$...$$` per formule display.

## Versione studenti: uso controllato dell'IA

Questo notebook mantiene la stessa struttura della versione docente: **15 tappe**, di cui **14 tappe didattiche** e **1 tappa di sintesi/esportazione**.

Alcune celle di codice sono state intenzionalmente sbiancate. Per ciascuna cella sbiancata è fornito un **prompt lecito**: il prompt può essere usato per ottenere supporto dall'IA nella ricostruzione locale del codice, ma lo studente resta responsabile di:

```text
1. esecuzione del codice;
2. verifica dei controlli numerici o logici;
3. interpretazione dell'output;
4. consegna del tracciato della chat, se l'IA è stata usata.
```

La struttura e la numerazione delle tappe coincidono con la versione docente.

### Nota tecnica per VS Code/Pylance

La cella di setup contiene alcune **annotazioni di variabile**. Servono solo a evitare falsi positivi di Pylance nelle celle sbiancate, dove le variabili verranno create progressivamente dallo studente durante la lezione.

Queste annotazioni hanno la forma:

```python
nome_variabile: tipo
```

e non sono assegnazioni. Quindi non sostituiscono il codice da completare e non rendono eseguibili le celle sbiancate prima che siano state completate.


### Nota tecnica su output e percorsi

Questa versione studenti è distribuita **senza output preesistenti**. Le immagini eventualmente visibili dopo l'apertura devono quindi essere prodotte solo dall'esecuzione del notebook studenti.

Tutte le figure e le tabelle generate dagli studenti sono salvate localmente dentro `05_CodiceSt/`:

```text
05_CodiceSt/
|-- graphics_Lez04_studenti/
`-- output_Lez04_studenti/
```

La cartella generale `graphics/` resta riservata alla versione docente e ai materiali ufficiali per manuale e slides.


## Struttura operativa

Il notebook è pensato per essere eseguito in VS Code/Jupyter.

Destinazioni previste:

```text
04_Codice/
`-- Lez04_probabilita_condizionamento_docente.ipynb

05_CodiceSt/graphics_Lez04_studenti/
|-- Cap04_istogramma_perdita.png
|-- Cap04_ecdf_perdita.png
|-- Cap04_quantili_perdita.png
|-- Cap04_medie_condizionate_stati.png
|-- Cap04_prob_superamento_soglie.png
|-- Cap04_distribuzioni_condizionate.png
|-- Cap04_sensibilita_M.png
`-- Cap04_partizioni_informative.png
```

Le tabelle CSV prodotte dagli studenti sono salvate in una sottocartella locale al notebook studenti:

```text
05_CodiceSt/output_Lez04_studenti/
```

Questa scelta mantiene il codice in `04_Codice/`, le figure in `05_CodiceSt/graphics_Lez04_studenti/`, e i file intermedi in una cartella separata.

## Tappa 0 — Intestazione, librerie e riproducibilità

**Obiettivo.** Fissare le impostazioni generali del modello computazionale.

**Controllo.** Verificare che $M>0$, $V_0>0$ e $\ell>0$.

**Interpretazione.** Il seed rende l'esperimento Monte Carlo riproducibile. Il parametro $M$ può essere ridotto a $50$ per una preview didattica, ma la versione docente usa un valore più alto per ottenere stime più stabili.

In [ ]:
# ============================================================
# Blocco 0.1 - Librerie di lavoro
# ============================================================
# Path serve per costruire percorsi di file e cartelle in modo robusto.
# E' preferibile a scrivere percorsi come semplici stringhe, perche' rende
# piu' chiaro dove vengono salvati grafici e tabelle.
from pathlib import Path

# NumPy e' la libreria di base per vettori, numeri casuali e calcoli numerici.
import numpy as np

# pandas permette di organizzare i dati in tabelle, chiamate DataFrame.
# Nel notebook useremo DataFrame per scenari simulati, frequenze e statistiche.
import pandas as pd

# matplotlib.pyplot e' la libreria usata per costruire i grafici.
import matplotlib.pyplot as plt

# ============================================================
# Blocco 0.2 - Impostazioni di visualizzazione
# ============================================================
# Queste opzioni non modificano i dati: regolano solo come pandas mostra
# le tabelle dentro Jupyter.
pd.set_option("display.precision", 6)
pd.set_option("display.max_columns", 20)

# ============================================================
# Blocco 0.3 - Cartelle di output
# ============================================================
# Path.cwd() restituisce la cartella da cui il notebook viene eseguito.
# In questo progetto il notebook si trova in 04_Codice/.
# Cartelle relative: il notebook studenti si trova in 05_CodiceSt/

NOTEBOOK_DIR = Path.cwd()

# Output separati dalla versione docente.
# Non usare MQF/graphics/ e non usare 04_Codice/output_Lez04/.
STUDENT_GRAPHICS_DIR = NOTEBOOK_DIR / "graphics_Lez04_studenti"
STUDENT_OUTPUT_DIR = NOTEBOOK_DIR / "output_Lez04_studenti"

# Alias usati nel resto del notebook: puntano a cartelle studenti.
GRAPHICS_DIR = STUDENT_GRAPHICS_DIR
OUTPUT_DIR = STUDENT_OUTPUT_DIR

GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Controlli espliciti sui percorsi della versione studenti.
assert GRAPHICS_DIR == NOTEBOOK_DIR / "graphics_Lez04_studenti"
assert OUTPUT_DIR == NOTEBOOK_DIR / "output_Lez04_studenti"
assert GRAPHICS_DIR.parent == NOTEBOOK_DIR
assert OUTPUT_DIR.parent == NOTEBOOK_DIR

# ------------------------------------------------------------
# Dichiarazioni per Pylance / VS Code
# ------------------------------------------------------------
# Le variabili seguenti sono create progressivamente nelle tappe del notebook.
# Le annotazioni servono solo a evitare falsi positivi reportUndefinedVariable
# nelle celle sbiancate. Non assegnano valori e non modificano l'esecuzione.

stati: np.ndarray
etichette: dict[str, str]
p: np.ndarray
mu: np.ndarray
sigma: np.ndarray
param_map: pd.DataFrame
seed: int
rng: np.random.Generator
M: int
V0: float
ell_base: float
soglie_base: np.ndarray
Z: np.ndarray
freq_emp: pd.DataFrame
quantile_levels: list[float]
soglia_grid: np.ndarray
valori_M: list[int]

df: pd.DataFrame
df_parametri: pd.DataFrame
df_frequenze: pd.DataFrame
df_stat_R_stato: pd.DataFrame
controllo_segno: pd.DataFrame
statistiche_L: pd.DataFrame
df_quantili: pd.DataFrame
df_soglie: pd.DataFrame
df_soglie_grid: pd.DataFrame
df_media: pd.DataFrame
df_cond: pd.DataFrame
df_valori_cond: pd.DataFrame
df_torre: pd.DataFrame
df_quantili_stato: pd.DataFrame
df_sens_M: pd.DataFrame
df_partizioni: pd.DataFrame
df_coerenza_partizioni: pd.DataFrame
df_controllo_file: pd.DataFrame

L_sorted: np.ndarray
ecdf: np.ndarray

E_L_hat: float
E_L_theory: float
diff_MC: float
E_E_L_cond_G_hat: float
tabella_generale: pd.DataFrame


## Tappa 1 — Definizione degli stati informativi di mercato

Si introduce una variabile discreta di stato

$$ Z\in\{N,V,S\},$$
dove $N$ indica un mercato normale, $V$ una fase di volatilità elevata e $S$ uno stato di stress.

Condizionatamente allo stato $Z=g$, il rendimento del portafoglio è simulato come$$R\mid Z=g \sim \mathcal{N}(\mu_g,\sigma_g^2).$$

**Controllo.** Le probabilità degli stati devono sommare a $1$, essere strettamente positive e le volatilità devono essere positive.

**Interpretazione.** Gli stati generano la partizione informativa rispetto alla quale saranno calcolate le previsioni condizionate.

### Prompt lecito — Tappa 1

```text
Sto costruendo un notebook Python per simulare rendimenti e perdite di portafoglio.
Devo definire tre stati informativi N, V, S con probabilità [0.70, 0.20, 0.10],
medie dei rendimenti [0.004, -0.006, -0.030] e volatilità [0.020, 0.045, 0.080].

Scrivi solo il codice Python locale per:
1. definire array numpy per stati, probabilità, medie e volatilità;
2. costruire un dizionario di etichette descrittive;
3. creare un DataFrame df_parametri;
4. costruire param_map con indice stato;
5. inserire assert che controllino somma delle probabilità, positività delle probabilità e positività delle volatilità.

Non scrivere codice per simulare stati, rendimenti o perdite.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 1
    # ============================================================

    # TODO: definire stati, etichette, probabilità, medie e volatilità.
# stati = ...
# etichette = ...
# p = ...
# mu = ...
# sigma = ...

# TODO: inserire i controlli sui parametri.
# assert ...

# TODO: costruire df_parametri e param_map.
# df_parametri = ...
# param_map = ...

# Output atteso:
df_parametri

## Tappa 2 — Simulazione degli stati informativi

Si simulano $M$ realizzazioni della variabile discreta $Z$.

La frequenza empirica dello stato $g$ è

$$\widehat{p}_g
=
\frac{1}{M}\sum_{k=1}^{M}\mathbf{1}_{\{Z^{(k)}=g\}}.$$

**Controllo.** Le frequenze empiriche devono sommare a $M$; con $M$ elevato devono essere vicine alle probabilità teoriche.

**Interpretazione.** La simulazione traduce la distribuzione teorica degli stati in scenari osservabili.

### Prompt lecito — Tappa 2

```text
Ho già definito in Python le variabili stati, p, df_parametri, M e rng.
Devo simulare M realizzazioni della variabile discreta Z usando rng.choice.
Poi devo costruire un DataFrame df con le colonne scenario e stato.
Infine devo costruire una tabella df_frequenze che confronti probabilità teoriche e frequenze empiriche.

Scrivi solo il codice per questa tappa.
Inserisci anche due controlli:
1. la somma delle frequenze deve essere M;
2. ogni stato deve avere frequenza positiva.

Non simulare rendimenti o perdite.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 2
    # ============================================================

    # TODO: simulare Z e costruire df con scenario e stato.
# Z = ...
# df = ...

# TODO: calcolare frequenze empiriche per stato.
# freq_emp = ...

# TODO: costruire df_frequenze con probabilità teoriche, frequenze relative e scarti.
# df_frequenze = ...

# TODO: controlli sulle frequenze.
# assert ...

# Output atteso:
df_frequenze

## Tappa 3 — Simulazione dei rendimenti condizionati agli stati

Per ogni scenario $k$, si assegna la coppia $(\mu_{Z^{(k)}},\sigma_{Z^{(k)}})$ e si simula

$$R^{(k)} \sim \mathcal{N}(\mu_{Z^{(k)}},\sigma_{Z^{(k)}}^2).$$

**Controllo.** Non devono esserci valori mancanti; le statistiche empiriche per stato devono essere ragionevolmente coerenti con i parametri teorici.

**Interpretazione.** La distribuzione non condizionata di $R$ è una miscela di distribuzioni condizionate agli stati.

### Prompt lecito — Tappa 3

```text
Ho un DataFrame df con una colonna stato. Ho anche param_map, con indice stato
e colonne mu_rendimento e sigma_rendimento. Devo:
1. associare a ogni riga di df il valore corretto di mu e sigma;
2. simulare il rendimento R usando rng.normal con loc e scale vettoriali;
3. costruire una tabella df_stat_R_stato con numerosità, media e deviazione standard empiriche per stato,
   più i parametri teorici mu e sigma;
4. inserire assert su lunghezza del DataFrame e assenza di valori mancanti.

Scrivi solo il codice di questa tappa.
Non costruire ancora la perdita L.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 3
    # ============================================================

    # TODO: associare mu e sigma a ciascuno scenario in base allo stato.
# df["mu"] = ...
# df["sigma"] = ...

# TODO: simulare i rendimenti condizionati agli stati.
# df["R"] = ...

# TODO: controllare lunghezza del DataFrame e assenza di valori mancanti.
# assert ...

# TODO: costruire df_stat_R_stato.
# df_stat_R_stato = ...

# Output atteso:
df_stat_R_stato

## Tappa 4 — Trasformazione dei rendimenti in perdite

La perdita monetaria del portafoglio è definita da

$$L=-V_0R.$$

**Controllo.** Se $R^{(k)}<0$, allora $L^{(k)}>0$; se $R^{(k)}>0$, allora $L^{(k)}<0$.

**Interpretazione.** La coda destra di $L$ rappresenta gli scenari sfavorevoli.

### Prompt lecito — Tappa 4

```text
Ho un DataFrame df con una colonna R di rendimenti simulati e un valore V0.
Devo costruire la perdita L = -V0 * R e una colonna booleana perdita_positiva.
Poi devo costruire una tabella controllo_segno che verifichi:
1. R < 0 implica L > 0;
2. R > 0 implica L < 0.

Inserisci anche un assert che verifichi che il controllo sul segno sia soddisfatto.
Scrivi solo il codice di questa tappa.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 4
    # ============================================================

    # TODO: costruire perdita e indicatore di perdita positiva.
# df["L"] = ...
# df["perdita_positiva"] = ...

# TODO: costruire controllo_segno e relativo assert.
# controllo_segno = ...
# assert ...

# Output atteso:
controllo_segno

## Tappa 5 — Distribuzione empirica della perdita

Si calcolano statistiche descrittive e quantili della variabile casuale simulata $L$.

**Controlli.**

- i quantili empirici devono essere ordinati;
- la funzione di ripartizione empirica deve essere non decrescente;
- l'istogramma deve essere coerente con la scala della perdita.

**Interpretazione.** La media descrive il centro della distribuzione; i quantili descrivono la coda destra, rilevante per il rischio.

### Prompt lecito — Tappa 5

```text
Ho un DataFrame df con una colonna L che rappresenta perdite simulate.
Devo costruire:
1. una tabella statistiche_L con count, mean, std, var, min, max;
2. una tabella df_quantili per i livelli [0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99];
3. un controllo che verifichi che i quantili siano ordinati;
4. i vettori L_sorted ed ecdf per la funzione di ripartizione empirica;
5. un controllo che ecdf sia non decrescente.

Scrivi solo il codice per calcoli e controlli. Non scrivere il codice dei grafici.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 5
    # ============================================================

    # TODO: costruire statistiche_L.
# statistiche_L = ...

# TODO: costruire df_quantili.
quantile_levels = [0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99]
# df_quantili = ...

# TODO: verificare ordinamento dei quantili.
# df_quantili["ordinamento_ok"] = ...
# assert ...

# TODO: costruire L_sorted ed ecdf e verificare monotonia.
# L_sorted = ...
# ecdf = ...
# assert ...

# Output atteso:
statistiche_L

In [ ]:
# Mostriamo separatamente la tabella dei quantili, per distinguere
# statistiche descrittive generali e informazioni sulla coda della perdita.
df_quantili

In [ ]:
# ============================================================
# Blocco 5.3 - Istogramma della perdita
# ============================================================
# L'istogramma mostra la distribuzione empirica della variabile L.
# bins controlla il numero di intervalli; edgecolor rende visibili le barre.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["L"], bins=50, edgecolor="black")

# Aggiungiamo due riferimenti: la media empirica e la soglia di perdita ell_base.
ax.axvline(df["L"].mean(), linestyle="--", label="media empirica")
ax.axvline(ell_base, linestyle=":", label=f"soglia {ell_base:g}")

# Etichette e titolo rendono il grafico interpretabile anche fuori dal notebook.
ax.set_xlabel("Perdita L")
ax.set_ylabel("Frequenza")
ax.set_title("Cap04 - Istogramma della perdita simulata")
ax.legend()
fig.tight_layout()

# Salviamo il grafico nella cartella locale studenti graphics_Lez04_studenti/.
fig_path = GRAPHICS_DIR / "Cap04_istogramma_perdita.png"
fig.savefig(fig_path, dpi=200)

fig_path

In [ ]:
# ============================================================
# Blocco 5.4 - Funzione di ripartizione empirica
# ============================================================
# Per costruire la funzione di ripartizione empirica ordiniamo le perdite.
L_sorted = np.sort(df["L"].to_numpy())

# ecdf[k] rappresenta la frazione di osservazioni minori o uguali
# al k-esimo valore ordinato.
ecdf = np.arange(1, M + 1) / M

# Una funzione di ripartizione deve essere non decrescente.
assert np.all(np.diff(ecdf) >= 0), "La funzione di ripartizione empirica non è non decrescente."

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.step(L_sorted, ecdf, where="post")
ax.axhline(0, color="black", linewidth=1.0)
ax.axvline(ell_base, linestyle=":", label=f"soglia {ell_base:g}")
ax.set_xlabel("Perdita L")
ax.set_ylabel("Frequenza cumulata empirica")
ax.set_title("Cap04 - Funzione di ripartizione empirica della perdita")
ax.legend()
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_ecdf_perdita.png"
fig.savefig(fig_path, dpi=200)

fig_path

In [ ]:
# ============================================================
# Blocco 5.5 - Istogramma con quantili di coda
# ============================================================
# Questo grafico riprende l'istogramma e mette in evidenza i quantili elevati,
# utili per ragionare sugli scenari di perdita piu' severi.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["L"], bins=50, edgecolor="black")

# Il ciclo for evita di ripetere tre volte lo stesso blocco di codice.
for q in [0.90, 0.95, 0.99]:
    q_value = df["L"].quantile(q)
    ax.axvline(q_value, linestyle="--", label=f"q{int(q*100)} = {q_value:.2f}")

ax.set_xlabel("Perdita L")
ax.set_ylabel("Frequenza")
ax.set_title("Cap04 - Quantili empirici della perdita")
ax.legend()
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_quantili_perdita.png"
fig.savefig(fig_path, dpi=200)

fig_path

## Tappa 6 — Probabilità empirica di superamento soglia

Per una soglia $\ell$, l'evento di interesse è

$$A_\ell=\{L>\ell\}.$$
Lo stimatore Monte Carlo è 
$$\widehat{\mathbb{P}}(L>\ell)
=
\frac{1}{M}
\sum_{k=1}^{M}
\mathbf{1}_{\{L^{(k)}>\ell\}}.$$

**Controlli.**

- le probabilità devono essere comprese tra $0$ e $1$;
- la probabilità di superamento deve essere non crescente al crescere della soglia.

**Interpretazione.** La soglia definisce quale porzione della distribuzione viene considerata finanziariamente critica.

### Prompt lecito — Tappa 6

```text
Ho un DataFrame df con colonna L e un array soglie_base.
Devo stimare, per ogni soglia ell in soglie_base, la probabilità empirica P(L > ell)
e il numero di superamenti. Voglio una tabella df_soglie con colonne:
soglia_l, prob_superamento_empirica, numero_superamenti.

Aggiungi due controlli:
1. ogni probabilità stimata deve essere tra 0 e 1;
2. le probabilità devono essere non crescenti quando la soglia aumenta.

Scrivi solo il codice per la tabella df_soglie e i controlli. Non costruire la griglia estesa delle soglie.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 6
    # ============================================================

    # TODO: costruire df_soglie con probabilità empiriche e numero di superamenti.
# df_soglie = ...

# TODO: controllare validità delle probabilità e monotonia rispetto alla soglia.
# df_soglie["probabilita_valida"] = ...
# df_soglie["monotonia_non_crescente"] = ...
# assert ...

# Output atteso:
df_soglie

In [ ]:
# ============================================================
# Blocco 6.3 - Curva di superamento per una griglia di soglie
# ============================================================
# linspace costruisce 61 soglie equidistanti tra 0 e 15.
soglia_grid = np.linspace(0, 15, 61)

# Ripetiamo la stima Prob(L > ell) su tutta la griglia di soglie.
df_soglie_grid = pd.DataFrame({
    "soglia_l": soglia_grid,
    "prob_superamento_empirica": [(df["L"] > ell).mean() for ell in soglia_grid]
})

# Controllo di monotonia della curva.
df_soglie_grid["monotonia_non_crescente"] = (
    df_soglie_grid["prob_superamento_empirica"].diff().fillna(0) <= 1e-12
)

assert df_soglie_grid["monotonia_non_crescente"].all(), "La curva di superamento soglia non è non crescente."

# Grafico della curva ell -> P(L > ell).
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df_soglie_grid["soglia_l"], df_soglie_grid["prob_superamento_empirica"], marker="o", markersize=3)
ax.axhline(0, color="black", linewidth=1.0)
ax.set_xlabel("Soglia di perdita")
ax.set_ylabel("Probabilità empirica di superamento")
ax.set_title("Cap04 - Probabilità empirica di superamento soglia")
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_prob_superamento_soglie.png"
fig.savefig(fig_path, dpi=200)

fig_path

## Tappa 7 — Valore atteso non condizionato della perdita

La stima Monte Carlo della perdita media non condizionata è

$$\widehat{\mathbb{E}}[L]
=
\frac{1}{M}\sum_{k=1}^{M}L^{(k)}.$$Nel modello parametrico scelto, la media teorica è$$\mathbb{E}[L]
=
-V_0\sum_g p_g\mu_g.$$

**Controllo.** La differenza tra media empirica e media teorica è errore Monte Carlo, non errore del modello.

**Interpretazione.** La media non condizionata aggrega tutti gli stati di mercato in un'unica previsione.

### Prompt lecito — Tappa 7

```text
Ho un DataFrame df con colonna L. Ho anche V0, p e mu.
Devo:
1. calcolare E_L_hat come media empirica di L;
2. calcolare E_L_theory = -V0 * somma_g p_g * mu_g;
3. calcolare la differenza Monte Carlo;
4. costruire df_media con righe E_hat[L], E_teorica[L] e differenza_MC.

Scrivi solo il codice di questa tappa.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 7
    # ============================================================

    # TODO: calcolare media empirica, media teorica e differenza Monte Carlo.
# E_L_hat = ...
# E_L_theory = ...
# diff_MC = ...

# TODO: costruire df_media.
# df_media = ...

# Output atteso:
df_media

## Tappa 8 — Valori attesi condizionati rispetto agli stati

Per ogni stato informativo $g$, si stima

$$\mathbb{E}[L\mid Z=g].$$
Lo stimatore empirico è
$$\widehat{\mathbb{E}}[L\mid Z=g]
=
\frac{1}{M_g}
\sum_{k:Z^{(k)}=g}
L^{(k)},
\qquad
M_g=\#\{k:Z^{(k)}=g\}.$$

**Controllo.** Ogni classe deve avere numerosità positiva.

**Interpretazione.** Il valore atteso condizionato modifica la previsione quando lo stato è noto; non modifica le perdite simulate.

### Prompt lecito — Tappa 8

```text
Ho un DataFrame df con colonne stato e L. Ho anche stati ed etichette.
Devo costruire df_cond usando groupby("stato") per ottenere, per ogni stato:
1. numerosità M_g;
2. media condizionata della perdita;
3. deviazione standard condizionata;
4. quantili 0.05, 0.50, 0.95 della perdita.

Poi devo riordinare le righe secondo l'array stati, aggiungere la descrizione dello stato,
riordinare le colonne e aggiungere un controllo classe_non_vuota.
Inserisci un assert che verifichi che ogni classe sia non vuota.

Scrivi solo il codice per tabella e controlli. Non scrivere il grafico.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 8
    # ============================================================

    # TODO: costruire df_cond con statistiche condizionate per stato.
# df_cond = ...

# TODO: aggiungere descrizione, riordinare colonne e controllare classi non vuote.
# df_cond["descrizione"] = ...
# df_cond = ...
# df_cond["classe_non_vuota"] = ...
# assert ...

# Output atteso:
df_cond

In [ ]:
# ============================================================
# Blocco 8.2 - Grafico delle medie condizionate
# ============================================================
# Il grafico confronta E[L | stato] con la media non condizionata E[L].
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(df_cond["descrizione"], df_cond["E_hat_L_cond"])
ax.axhline(0, color="black", linewidth=1.0)
ax.axhline(df["L"].mean(), linestyle="--", label="media non condizionata")
ax.set_xlabel("Stato informativo")
ax.set_ylabel("Media empirica condizionata di L")
ax.set_title("Cap04 - Medie condizionate per stato")
ax.tick_params(axis="x", rotation=20)
ax.legend()
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_medie_condizionate_stati.png"
fig.savefig(fig_path, dpi=200)

fig_path

## Tappa 9 — Costruzione della variabile $\mathbb{E}[L\mid\mathcal{G}]$

La sigma-algebra informativa è generata dalla partizione degli stati:

$$\mathcal{G}=\sigma(A_N,A_V,A_S).$$
La variabile condizionata è
$$\mathbb{E}[L\mid\mathcal{G}]
=
\sum_g \mathbb{E}[L\mid Z=g]\mathbf{1}_{\{Z=g\}}.$$

**Controllo.** La variabile $\mathbb{E}[L\mid\mathcal{G}]$ deve essere costante all'interno di ciascuno stato.

**Interpretazione.** Questa previsione condizionata non è un numero: è una variabile casuale che assume valori diversi a seconda dello stato informativo.

### Prompt lecito — Tappa 9

```text
Ho un DataFrame df con colonne stato e L, e una tabella df_cond con colonne stato
ed E_hat_L_cond. Devo costruire la variabile simulata E[L | G]:
1. creare una Series cond_map che associa a ogni stato la sua media condizionata;
2. creare in df la colonna E_L_cond_G usando map;
3. costruire una tabella df_valori_cond che, per ogni stato, riporti il numero
   di valori distinti di E_L_cond_G e il valore assunto;
4. aggiungere un controllo costante_nello_stato;
5. inserire un assert che verifichi che il valore sia costante dentro ogni stato.

Scrivi solo il codice di questa tappa.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 9
    # ============================================================

    # TODO: costruire la mappa stato -> media condizionata.
# cond_map = ...

# TODO: costruire la variabile E_L_cond_G.
# df["E_L_cond_G"] = ...

# TODO: verificare che E_L_cond_G sia costante entro ciascuno stato.
# df_valori_cond = ...
# df_valori_cond["costante_nello_stato"] = ...
# assert ...

# Output atteso:
df_valori_cond

## Tappa 10 — Controllo della proprietà a torre (Valore atteso condizionato iterato)

Si verifica numericamente la proprietà

$$\mathbb{E}\left[\mathbb{E}[L\mid\mathcal{G}]\right]
=
\mathbb{E}[L].$$

**Controllo.** La differenza tra le due medie empiriche deve essere nulla o trascurabile.

**Interpretazione.** Il condizionamento differenzia le previsioni tra stati, ma conserva la coerenza con la media aggregata.

### Prompt lecito — Tappa 10

```text
Ho un DataFrame df con colonne L ed E_L_cond_G. Ho già calcolato E_L_hat come media empirica di L.
Devo verificare numericamente la proprietà a torre E[E[L | G]] = E[L].

Scrivi codice Python per:
1. calcolare E_E_L_cond_G_hat come media della colonna E_L_cond_G;
2. costruire una tabella df_torre con media_empirica_L, media_empirica_E_L_cond_G
   e differenza_assoluta;
3. inserire un assert che controlli che la differenza sia minore di 1e-10.

Scrivi solo il codice di questa tappa.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 10
    # ============================================================

    # TODO: calcolare la media empirica di E[L | G].
# E_E_L_cond_G_hat = ...

# TODO: costruire df_torre.
# df_torre = ...

# TODO: controllo numerico della proprietà a torre.
# assert ...

# Output atteso:
df_torre

## Tappa 11 — Distribuzione non condizionata e distribuzioni condizionate

Si confronta la distribuzione complessiva della perdita con le distribuzioni condizionate agli stati.

**Controllo.** I quantili condizionati devono essere coerenti con la parametrizzazione: lo stato di stress dovrebbe presentare valori di coda più severi.

**Interpretazione.** La distribuzione non condizionata può nascondere eterogeneità rilevante tra stati informativi.

### Prompt lecito — Tappa 11

```text
Ho un DataFrame df con colonne stato e L. Ho anche l'array stati e il dizionario etichette.
Devo costruire una tabella df_quantili_stato che riporti, per ogni stato,
i quantili 0.05, 0.50, 0.95 e 0.99 della perdita.
La tabella deve essere ordinata secondo l'array stati e deve includere la descrizione dello stato.

Scrivi solo il codice per costruire la tabella df_quantili_stato.
Non scrivere il codice del grafico.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 11
    # ============================================================

    # TODO: costruire df_quantili_stato con quantili per stato.
# df_quantili_stato = ...

# Output atteso:
df_quantili_stato

In [ ]:
# ============================================================
# Blocco 11.2 - Confronto grafico tra distribuzione non condizionata
# e distribuzioni condizionate
# ============================================================
fig, ax = plt.subplots(figsize=(7, 4.5))

# Istogramma della perdita non condizionata: usa tutti gli scenari.
ax.hist(
    df["L"],
    bins=60,
    density=True,
    histtype="step",
    linewidth=2,
    label="non condizionata"
)

# Istogrammi condizionati: uno per ciascuno stato informativo.
for s in stati:
    ax.hist(
        df.loc[df["stato"] == s, "L"],
        bins=60,
        density=True,
        histtype="step",
        linewidth=1.5,
        label=etichette[s]
    )

ax.set_xlabel("Perdita L")
ax.set_ylabel("Densità empirica")
ax.set_title("Cap04 - Distribuzioni della perdita per stato")
ax.legend()
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_distribuzioni_condizionate.png"
fig.savefig(fig_path, dpi=200)

fig_path

## Tappa 12 — Sensibilità al numero di simulazioni

Una stima Monte Carlo è una quantità casuale. Si ripete la stima per diversi valori di $M$.

**Controllo.** La variabilità delle stime tende a ridursi al crescere di $M$, senza richiedere monotonia perfetta.

**Interpretazione.** La numerosità degli scenari è parte della qualità computazionale del risultato.

### Prompt lecito — Tappa 12

```text
Partendo dalle variabili già definite stati, p, param_map, V0 e seed,
devo scrivere una funzione simula_sintesi(M_local, seed_local) che:
1. simuli gli stati;
2. associ mu e sigma allo stato;
3. simuli i rendimenti;
4. calcoli le perdite;
5. restituisca un dizionario con M, media empirica di L, probabilità P(L > 5)
   e deviazione standard di L.

Poi devo applicare la funzione ai valori M = [50, 100, 500, 1000, 5000, 10000, 50000]
e costruire df_sens_M.

Scrivi solo il codice per questa tappa.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 12
    # ============================================================

    # TODO: definire la funzione simula_sintesi(M_local, seed_local).
# def simula_sintesi(M_local: int, seed_local: int) -> dict:
#     ...

# TODO: applicare la funzione a diversi valori di M.
# valori_M = ...
# df_sens_M = ...

# Output atteso:
df_sens_M

In [ ]:
# ============================================================
# Blocco 12.3 - Grafico della sensibilita' a M
# ============================================================
# Rappresentiamo la stima della media al variare del numero di simulazioni.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df_sens_M["M"], df_sens_M["E_hat_L"], marker="o")
ax.axhline(0, color="black", linewidth=1.0)
ax.axhline(E_L_theory, linestyle="--", label="media teorica")
ax.set_xlabel("Numero di simulazioni M")
ax.set_ylabel("Stima della media di L")
ax.set_title("Cap04 - Sensibilità della media Monte Carlo a M")

# La scala logaritmica rende leggibili valori di M molto diversi tra loro.
ax.set_xscale("log")
ax.legend()
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_sensibilita_M.png"
fig.savefig(fig_path, dpi=200)

fig_path

## Tappa 13 — Sensibilità alla soglia di perdita

Si analizza la funzione empirica

$$\ell \mapsto \widehat{\mathbb{P}}(L>\ell).$$

**Controllo.** La funzione deve essere non crescente.

**Interpretazione.** La soglia determina quale area della distribuzione viene letta come evento di rischio.

### Prompt lecito — Tappa 13

```text
Ho un DataFrame df con colonna L. Devo costruire una griglia di soglie tra 0 e 15
e stimare per ogni soglia la probabilità empirica P(L > ell).
Voglio una tabella df_soglie_grid con colonne soglia_l e prob_superamento_empirica.
Aggiungi un controllo che la probabilità sia non crescente rispetto alla soglia.

Scrivi solo il codice per tabella e controllo.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 13
    # ============================================================

    # TODO: costruire griglia di soglie e probabilità empiriche di superamento.
# soglia_grid = ...
# df_soglie_grid = ...

# TODO: controllo di monotonia.
# df_soglie_grid["monotonia_non_crescente"] = ...
# assert ...

# Output atteso:
df_soglie_grid.head(10)

## Tappa 14 — Partizione informativa grossolana e partizione informativa fine

Si confrontano due livelli informativi:

$$\mathcal{G}_1=\sigma(\text{normale},\text{non normale})$$e$$\mathcal{G}_2=\sigma(\text{normale},\text{volatilità elevata},\text{stress}).$$

**Controllo.** Entrambe le previsioni condizionate devono rispettare la coerenza aggregata.

**Interpretazione.** Una partizione più fine produce previsioni più differenziate, ma non altera la media non condizionata.

### Prompt lecito — Tappa 14

```text
Ho un DataFrame df con colonne stato, L ed E_L_cond_G. La colonna stato distingue N, V, S.
Devo costruire una partizione grossolana G1 con due classi:
normale se stato == "N", non normale altrimenti.

Poi devo:
1. stimare E[L | G1];
2. usare E_L_cond_G come previsione rispetto alla partizione fine G2;
3. costruire df_partizioni che confronti numerosità e media condizionata per G1 e G2;
4. costruire df_coerenza_partizioni con E_hat[L], E_hat[E[L|G1]] ed E_hat[E[L|G2]];
5. verificare con assert che le differenze dalla media di L siano nulle o trascurabili.

Scrivi solo il codice per tabelle e controlli. Non scrivere il grafico.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 14
    # ============================================================

    # TODO: costruire stato_grossolano.
# df["stato_grossolano"] = ...

# TODO: stimare E[L | G1] e recuperare E[L | G2].
# cond_gross = ...
# df["E_L_cond_G1"] = ...
# df["E_L_cond_G2"] = ...

# TODO: costruire df_partizioni.
# df_partizioni = ...

# TODO: costruire df_coerenza_partizioni e relativo controllo.
# df_coerenza_partizioni = ...
# assert ...

# Output atteso:
df_partizioni

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 14
    # ============================================================

    # TODO: costruire stato_grossolano.
# df["stato_grossolano"] = ...

# TODO: stimare E[L | G1] e recuperare E[L | G2].
# cond_gross = ...
# df["E_L_cond_G1"] = ...
# df["E_L_cond_G2"] = ...

# TODO: costruire df_partizioni.
# df_partizioni = ...

# TODO: costruire df_coerenza_partizioni e relativo controllo.
# df_coerenza_partizioni = ...
# assert ...

# Output atteso:
df_partizioni

In [ ]:
# ============================================================
# Blocco 14.4 - Grafico del confronto tra partizioni
# ============================================================
# copy evita di modificare accidentalmente df_partizioni mentre prepariamo
# etichette piu' adatte al grafico.
plot_part = df_partizioni.copy()

# Estraiamo G1 o G2 dal nome della partizione per costruire etichette compatte.
prefisso = plot_part["partizione"].str.extract(r"(G\d)", expand=False).fillna("")
plot_part["etichetta"] = prefisso + " - " + plot_part["classe"].astype(str)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(plot_part["etichetta"], plot_part["E_hat_L_cond"])
ax.axhline(0, color="black", linewidth=1.0)
ax.axhline(df["L"].mean(), linestyle="--", label="media non condizionata")
ax.set_xlabel("Classe informativa")
ax.set_ylabel("Media condizionata di L")
ax.set_title("Cap04 - Partizione grossolana e partizione fine")
ax.tick_params(axis="x", rotation=25)
ax.legend()
fig.tight_layout()

fig_path = GRAPHICS_DIR / "Cap04_partizioni_informative.png"
fig.savefig(fig_path, dpi=200)

fig_path

## Tappa 15 — Esportazione ordinata di dati, tabelle e figure

**Obiettivo.** Rendere il prodotto computazionale riproducibile.

**Controllo.** I file devono essere creati e non devono essere vuoti.

**Interpretazione.** Il notebook produce oggetti riutilizzabili per manuale e slides: dati simulati, tabelle e figure.

### Prompt lecito — Tappa 15

```text
Ho già prodotto le tabelle principali del notebook e una lista di figure attese.
Devo esportare le tabelle in CSV nella cartella OUTPUT_DIR e controllare che tabelle e figure esistano e non siano vuote.
Scrivi codice Python per:
1. costruire un dizionario tabelle;
2. esportare ogni tabella in CSV;
3. costruire una lista figure_attese;
4. creare df_controllo_file con tipo, file, esistenza e dimensione;
5. aggiungere un controllo non_vuoto;
6. inserire assert su esistenza e non vacuità.

Scrivi solo il codice di esportazione e controllo.
```

In [ ]:
# ============================================================
    # SPAZIO DA COMPLETARE — Tappa 15
    # ============================================================

    # TODO: costruire dizionario tabelle ed esportare in CSV.
# tabelle = ...
# for nome_file, tabella in tabelle.items():
#     ...

# TODO: definire figure_attese e costruire df_controllo_file.
# figure_attese = ...
# controllo_file = ...
# df_controllo_file = ...

# TODO: controllare esistenza e non vacuità dei file.
# df_controllo_file["non_vuoto"] = ...
# assert ...

# Output atteso:
df_controllo_file

## Dichiarazione d'uso dell'IA

Compilare questa sezione nel notebook consegnato.

```text
Ho usato strumenti di IA? [sì/no]

Numero totale di prompt usati:

Prompt usati:
1.
2.
3.
4.
5.

Per ciascun prompt indicare:
- tappa a cui si riferisce;
- obiettivo del prompt;
- se il suggerimento dell'IA è stato modificato;
- quale controllo numerico/logico è stato verificato.

Dichiaro che il commento interpretativo finale è stato scritto da me
e che ho verificato personalmente tutti i controlli richiesti.
```

## Sintesi interpretativa

Il notebook realizza il seguente percorso:

$$\text{stato informativo}
\longrightarrow
\text{rendimento}
\longrightarrow
\text{perdita}
\longrightarrow
\text{distribuzione empirica}
\longrightarrow
\text{evento}
\longrightarrow
\text{previsione condizionata}.$$Il punto concettuale da enfatizzare in aula è che il valore atteso condizionato rispetto a $\mathcal{G}$ non è un singolo numero, ma una variabile casuale costante sugli elementi della partizione informativa.

La proprietà 
$$\mathbb{E}\left[\mathbb{E}[L\mid\mathcal{G}]\right]
=
\mathbb{E}[L]$$

è verificata numericamente dalla costruzione stessa della colonna `E_L_cond_G`.